# Adike Dhara - Exploratory Data Analysis

Explore arecanut price trends, seasonality, and missing days for Karnataka mandis.

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
from src.features import load_processed, build_features

PROCESSED = Path("data/processed")
files = sorted(PROCESSED.glob("*_daily.csv"))
print("Processed varieties:", [f.stem.replace("_daily","") for f in files])

In [ ]:
# Load one variety for exploration
df = load_processed(PROCESSED / "Moati_daily.csv")
print(df.shape)
print(df.head())
print(df["variety"].value_counts())
print("Missing days flagged:", int((df["missing_day"]==True).sum()))
print("Outliers flagged:", int((df["outlier"]==True).sum()))

## Price trend by market

In [ ]:
fig = px.line(
    df, x="date", y="modal_price", color="market",
    title="Arecanut modal price over time - Moati",
    labels={"modal_price": "Modal price (₹/quintal)", "date": "Date"},
)
fig.show()

## Seasonality & missing days

Average price by day-of-week and month, plus missing-day calendar coverage.

In [ ]:
df2 = df.dropna(subset=["modal_price"]).copy()
df2["dow"] = df2["date"].dt.day_name()
df2["month"] = df2["date"].dt.month_name()

dow_avg = df2.groupby("dow")["modal_price"].mean().reindex([
    "Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"
])
print("Avg price by day of week:")
print(dow_avg)

print("\nMissing days per market:")
print(df.groupby("market")["missing_day"].sum().sort_values(ascending=False))

In [ ]:
## Check features for leakage
X, y = build_features(df)
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Any NaN in X:", X.isna().any().any())
print(X.head())